# DGCluster Evaluation


Esse notebook tem por objetivo reproduzir os experimentos da seção de uso do SynCo para avaliação paramétrica de algoritmos de agrupamento.

Vamos criar um grafo com a seguinte configuração:

Para a construção do número de vértices e dos vértices por classe:

$\mathbf{y} = [150, 200, 300, 100, 50, 250, 150]$

Para as arestas:

* Se o cluster contém entre 1 e 99 vértices, então terá 4x a quantidade de arestas
* Se o cluster contém entre 100 e 199 vértices, então terá 3x a quantidade de arestas
* Se o cluster contém mais que 200 vértices, então terá 2.3x a quantidade de arestas

Para as distribuições entre grupos:

* Se o cluster contém entre 1 e 99 vértices, sua distribuição é uniforme
* Se o cluster contém entre 100 e 199 vértices, sua distribuição é normal
* Se o cluster contém mais que 200 vértices, sua distribuição é lei de potência

Para a distribuição de interações entre subcomunidades $C$:

* Se o cluster contém entre 1 e 99 vértices, não tem subcomunidades
* Se o cluster contém entre 100 e 199 vértices, então tem 2 subcomunidades, onde a probabilidade da ligação de $i$ para $i$ é 0.8, enquanto de $i$ para $j$ é de 0.2
* Se o cluster contém mais de 199 vértices, então tem 3 subcomunidades, onde a probabilidade da ligação de $i$ para $i$ é 0.99, enquanto de $i$ para $j$ é de 0.01, para ambas as outras subcomunidades

Para a distribuição de subcomunidades $M$:

* Se o cluster $i$ tem entre 1 e 99 vértices, $M_i = [1]$
* Se o cluster $i$ tem entre 100 e 199 vértices, $M_i = [0.75, 0.25]$
* Se o cluster $i$ tem mais de 200 vértices, $M_i = [0.33, 0.33, 0.33]$

A matriz de ruído $$N = \begin{bmatrix} 0 & 0.2 & 0.8 \\ 0.3 & 0 & 0.7 \\ 0.5 & 0.5 & 0 \\ \end{bmatrix}$$

O valor de $\rho = 0.05$

Essa configuração garante a seguinte propriedade ao grafo gerado pelo SynCo: "Grupos menores não tem características de subcomunidade, seu grau médio é alto e tem desvio padrão baixo. Comunidades de tamanho médio tem características de subcomunidades com mais força, seu grau médio é menor e tem desvio padrão um pouco maior. Comunidades de tamanho grande tem características de lei de potência."

Utilizando o algoritmo DGCluster vamos buscar responder a seguinte pergunta:

"O parâmetro $\lambda$ que controla a perda por modularidade (modularity loss) deve ser maior ou menor para lidar com grafos nesse caso?"

# Bibliotecas

In [ ]:
import sys
sys.path.append("..")

import torch

from synco import SynCoGenerator

# Eurística

In [ ]:
def euristic_edges(x):
    if 1 < x < 99:
        return 3
    if 100 < x < 199:
        return 3
    else:
        return 3
    
def euristic_distribution(x):
    if 1 < x < 99:
        return "uniform"
    if 100 < x < 199:
        return "normal"
    else:
        return "power_law"
    
def euristic_subcomm_interaction(x):
    if 1 < x < 99:
        return torch.tensor([1])
    if 100 < x < 199:
        return torch.tensor([[0.8,0.2], [0.2,0.8]])
    else:
        return torch.tensor([[0.99,0.01,0.01], [0.01, 0.99, 0.01], [0.01,0.01,0.99]])
    
def euristic_subcomm_prob(x):
    if 1 < x < 99:
        return torch.tensor([1])
    if 100 < x < 199:
        return torch.tensor([0.75, 0.25])
    else:
        return torch.tensor([0.33, 0.33, 0.33])


# Definição do Grafo SynCo

In [ ]:
y = torch.tensor([150, 200, 300, 100, 50, 250, 150])
# d = [normal, pl, pl, normal, uniform, pl, normal]
e = torch.tensor([euristic_edges(x)*x for x in y])
k= len(y)
d = [euristic_distribution(x) for x in y]
C = [euristic_subcomm_interaction(x) for x in y]
M = [euristic_subcomm_prob(x) for x in y]
# N = torch.tensor([[0 , 0.2 , 0.11],[0.3 , 0 , 0.7], [0.5 , 0.5, 0] ])
N = torch.ones(size = (len(y), len(y))) - torch.eye(n = len(y))



graph = SynCoGenerator().generate(num_nodes=y.sum().item(), y = y, k = k, e = e, C = C, d = d, rho = 0.01, N = N, M = M, alpha_powerlaw=1.2, dimensions = 256, alpha = 0.2, sigma = 0.1)

In [ ]:
N

In [ ]:
graph

In [ ]:
N = torch.tensor([[0, 0.4, 0], [0.4, 0, 0.6], [0, 0.6, 0]])

N.sum(dim=(1))

# Running DGCluster

In [ ]:
import subprocess
import os

rgs = [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
rgs2 = [0.1, 0.5, 0.8]
rho_values = [0.0, 0.01, 0.02]

for scatt_alpha in rgs2:
    for scatt_lbd in rgs2:
        for scatt_rho in rho_values:
            path = f"DGCluster_SynCo_datasets/scatt_alpha_{scatt_alpha}_lbd_{scatt_lbd}_rho_{scatt_rho}.pt"
            if not os.path.exists(path):
                graph = SynCoGenerator().generate(num_nodes=y.sum().item(), y = y, k = k, e = e, C = C, d = d, rho = scatt_rho, N = N, M = M, alpha_powerlaw=1.2, dimensions = 256, alpha = scatt_alpha, sigma = scatt_lbd)
                dt = graph.to_data_pytorch()
                torch.save(dt, path)
            else:
                print(f"Graph Alredy Exists in {path}")
            for lbd in rgs:
                for seed in [0,1,2,3,4]:
                    subprocess.run([
                        "python3", "../DGCluster/main.py",
                        "--dataset", "custom",
                        "--custom_dataset_path", str(path),
                        "--lam", str(lbd),
                        "--epochs", "100",
                        "--SCAtt_lbd", str(scatt_lbd),
                        "--SCAtt_alpha", str(scatt_alpha),
                        "--SCAtt_rho", str(scatt_rho),
                        "--seed", str(seed)
                    ], check=True)

In [ ]:
rst = torch.load('results_DGCluster/results_custom_0.6_100_gcn_0_0.8_0.5_0.0.pt')

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame() 

for path in os.listdir('results_DGCluster/'):
    rst = torch.load(f'results_DGCluster/{path}')
    rst["lambda_DGCluster"] = path.split('_')[2]
    tmp_df = pd.DataFrame([rst])
    df = pd.concat([df, tmp_df], ignore_index=True)

In [ ]:
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("DGCluster_results.csv")

# Garantir que são numéricas (evita comparação errada)
df["SCAtt_rho"] = pd.to_numeric(df["SCAtt_rho"], errors="coerce")
df["SCAtt_alpha"] = pd.to_numeric(df["SCAtt_alpha"], errors="coerce")
df["lambda_DGCluster"] = pd.to_numeric(df["lambda_DGCluster"], errors="coerce")
df["nmi"] = pd.to_numeric(df["nmi"], errors="coerce")

values = [0.1, 0.5, 0.8]  # agora numérico

plt.figure(figsize=(8, 5))

for vl in values:
    sub = df[
        (np.isclose(df["SCAtt_rho"], 0.01)) &
        (np.isclose(df["SCAtt_alpha"], vl))
    ]

    curve = (
        sub.groupby("lambda_DGCluster", as_index=False)["num_clusters"]
           .mean()
           .sort_values("lambda_DGCluster")
    )
    # curve = sub[sub["SCAtt_lbd"] == 0.5].sort_values("lambda_DGCluster")

    plt.plot(curve["lambda_DGCluster"], curve["num_clusters"], marker="o", label=f"SCAtt_alpha={vl}")

plt.xlabel("lambda_DGCluster")
plt.ylabel("Mean NMI")
plt.title("NMI vs lambda_DGCluster (SCAtt_rho=0.01)")
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("DGCluster_results.csv")

# Garantir que são numéricas (evita comparação errada)
df["SCAtt_rho"] = pd.to_numeric(df["SCAtt_rho"], errors="coerce")
df["SCAtt_alpha"] = pd.to_numeric(df["SCAtt_alpha"], errors="coerce")
df["lambda_DGCluster"] = pd.to_numeric(df["lambda_DGCluster"], errors="coerce")
df["nmi"] = pd.to_numeric(df["nmi"], errors="coerce")

values = [0.1, 0.5, 0.8]  # agora numérico

plt.figure(figsize=(8, 5))

for vl in values:
    sub = df[
        (np.isclose(df["SCAtt_rho"], 0.01)) &
        (np.isclose(df["SCAtt_lbd"], vl))
    ]

    curve = (
        sub.groupby("lambda_DGCluster", as_index=False)["num_clusters"]
           .mean()
           .sort_values("lambda_DGCluster")
    )
    # curve = sub.sort_values("lambda_DGCluster")

    plt.plot(curve["lambda_DGCluster"], curve["num_clusters"], marker="o", label=f"SCAtt_lbd={vl}")

plt.xlabel("lambda_DGCluster")
plt.ylabel("Mean NMI")
plt.title("NMI vs lambda_DGCluster (SCAtt_rho=0.01)")
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("DGCluster_results.csv")

# Garantir que são numéricas (evita comparação errada)
df["SCAtt_rho"] = pd.to_numeric(df["SCAtt_rho"], errors="coerce")
df["SCAtt_alpha"] = pd.to_numeric(df["SCAtt_alpha"], errors="coerce")
df["SCAtt_lbd"] = pd.to_numeric(df["SCAtt_lbd"], errors="coerce")
df["lambda_DGCluster"] = pd.to_numeric(df["lambda_DGCluster"], errors="coerce")
df["nmi"] = pd.to_numeric(df["nmi"], errors="coerce")

values = [0.1, 0.5, 0.8]  # agora numérico

plt.figure(figsize=(8, 5))

for vl in values:
    sub = df[
        (np.isclose(df["SCAtt_rho"], 0.01)) &
        (np.isclose(df["SCAtt_lbd"], vl))
    ]

    curve = (
        sub.groupby("lambda_DGCluster", as_index=False)["nmi"]
           .mean()
           .sort_values("lambda_DGCluster")
    )

    plt.plot(curve["lambda_DGCluster"], curve["nmi"], marker="o", label=f"SCAtt_lbd={vl}")

plt.xlabel("lambda_DGCluster")
plt.ylabel("Mean NMI")
plt.title("NMI vs lambda_DGCluster (SCAtt_rho=0.01)")
plt.legend()
plt.show()

In [ ]:

df = pd.read_csv("DGCluster_results.csv")

# Garantir que são numéricas (evita comparação errada)
df["SCAtt_rho"] = pd.to_numeric(df["SCAtt_rho"], errors="coerce")
df["SCAtt_alpha"] = pd.to_numeric(df["SCAtt_alpha"], errors="coerce")
df["lambda_DGCluster"] = pd.to_numeric(df["lambda_DGCluster"], errors="coerce")
df["nmi"] = pd.to_numeric(df["nmi"], errors="coerce")

values = [0.1, 0.5, 0.8]  # agora numérico

for alph in values:
    for lbd in values:
        tmp_df = df[(df["SCAtt_alpha"] == alph) & (df["SCAtt_lbd"] == lbd) & (df['SCAtt_rho'] == 0.02)][["nmi", "lambda_DGCluster"]].sort_values(by = "lambda_DGCluster")
        plt.plot(tmp_df["lambda_DGCluster"], tmp_df["nmi"])


In [ ]:
df["lambda_DGCluster"].unique()